<a href="https://colab.research.google.com/github/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/ct_seg_to_tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validated CT/DICOM SEG tensors and patient-disjoint splits

Updated: October 2026

## Goal

Turn a small public **C4KC-KiTS** cohort into traceable PyTorch inputs: an HU image
`[1, D, H, W]`, separate binary mask channels `[K, D, H, W]`, an affine in DICOM
patient coordinates, and a patient-disjoint train/test manifest.

The [IDC Hugging Face dataset](https://huggingface.co/datasets/ImagingDataCommons/idc-index-data)
is a **metadata catalog**, not pixel data. Its `train` split is a Hub convention,
not a medical ML training partition. This notebook queries a pinned catalog and
uses `idc-index` to download only selected DICOM series. No cloud account or GPU
is required. Start with [Getting Started Part 2](https://github.com/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/getting_started/part2_searching_basics.ipynb)
if you are new to IDC.

### Scope and assumptions

- Four patients, selected deterministically by the smallest combined CT/SEG download.
  This deliberately biased convenience sample is for learning, not model evaluation.
- Legacy single-frame CT on a regular grid, linear rescaling to HU, and one BINARY
  SEG instance per series. Enhanced CT, non-regular spacing, gantry tilt, and
  resampled masks are rejected. Nothing is silently resampled.
- `seg_index.segmented_SeriesInstanceUID` selects the CT, and
  `volume_geometry_index.regularly_spaced_3d_volume` filters it before download.
  `highdicom` loads HU volumes and aligns separate SEG channels without resampling.
  Source SOP Instance UIDs and SEG-to-source frame correspondence remain local
  integrity checks; a patient/study match is insufficient.
- Separate channels retain overlaps. Coded category/type, modifiers and anatomic
  region define a stable channel order across cases; free-text labels and local
  segment numbers alone do not define classes.
- Missing CT files are errors. Omitted mask frames are also errors unless the
  caller explicitly enables an empty-frame policy for a verified dataset.
- The instructional split is within the public KiTS19 training collection; it is
  not the official challenge test set. Cross-collection identity reconciliation
  and statistically representative sampling are outside this example.

## Setup

Install the versions used for this example in a fresh Python 3.11/3.12 environment.
The PyTorch wheel is CPU-only. Local Jupyter users also need `ipykernel` and
`nbclient` to execute the notebook programmatically.

In [ ]:
import subprocess
import sys

# check_call stops execution if either installation fails.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "idc-index==0.13.0",
    "pydicom==3.0.2", "highdicom==0.28.2", "numpy==2.3.5",
    "pandas==2.2.3", "matplotlib==3.10.8",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "torch==2.14.1",
    "--index-url", "https://download.pytorch.org/whl/cpu",
])

### Parameters

The default uses Hugging Face revision `93983d810bea0454ae1a400ab62c48dd01f8d0ca`
of the IDC v25 catalog. Set `CATALOG_SOURCE = "idc-index"` to use the corresponding
package indices instead. Both routes are checked for matching release metadata.

Metadata downloads are separate from DICOM download size estimates. All paths
below are relative to the working directory. Rerunning reuses existing DICOM
files and replaces generated outputs for the same sample.

In [ ]:
from importlib.metadata import version
from pathlib import Path
from time import perf_counter
import json
import hashlib

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import requests
import torch
from idc_index import IDCClient
from IPython.display import display

CATALOG_SOURCE = "huggingface"   # or "idc-index"
HF_REVISION = "93983d810bea0454ae1a400ab62c48dd01f8d0ca"
COLLECTION = "c4kc_kits"
N_PATIENTS = 4
SPLIT_SEED = 2026
TEST_FRACTION = 0.25
MAX_DICOM_MB = 200.0
ALLOW_OMITTED_EMPTY_FRAMES = False
WORK_DIR = Path("idc_ct_seg_data")
DICOM_DIR = WORK_DIR / "dicom"
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_started = perf_counter()

## Steps

### 1. Use the catalog and highdicom at their intended boundaries

The [IDC index reference](https://idc-index.readthedocs.io/en/latest/indices_reference.html)
and [tensor-loading example](https://huggingface.co/datasets/ImagingDataCommons/idc-index-data#loading-images-as-tensors)
provide the building blocks used below:

- `seg_index` supplies CT/SEG correspondence, without reconstructing it manually.
- `volume_geometry_index` excludes CT series that are not regular 3D grids
  before any DICOM download.
- `hd.get_volume_from_series()` orders CT slices, checks the actual downloaded
  geometry, applies the modality transform to HU, and retains the LPS affine.
- `seg.get_volume(combine_segments=False).match_geometry(ct_volume)` aligns
  separate binary channels by padding/flipping/permuting; it does not interpolate.

The remaining local guards check download completeness, declared HU units,
source UIDs, and each SEG frame's correspondence to its referenced CT. The CT
geometry index does not establish those pair-specific properties. C4KC-KiTS
omits `SpatialLocationsPreserved`; absence is accepted only after these checks,
while explicit `NO` is rejected. SEG position tolerance is 0.01 mm and orientation
and pixel-spacing tolerance is `1e-4`. Highdicom CT spacing tolerance is 0.01 mm;
the final grid match uses `1e-4` voxels to tolerate rounding only.

The exported affine maps the library's array indices `[D, H, W]` to **DICOM LPS
millimeters**. Slice direction comes from highdicom and may differ from file or
InstanceNumber order. No display window or real-world-value transform is applied
to the saved HU image. HU units must be explicit or guaranteed by the
[original CT rule in DICOM PS3.3 C.8.2.1](https://dicom.nema.org/medical/Dicom/current/output/chtml/part03/sect_C.8.2.html);
missing units on derived/localizer/multi-energy images are not assumed to be HU.

### Validation helpers

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import random
from pathlib import Path

import pandas as pd
import highdicom as hd
import numpy as np
import pydicom
from pydicom.uid import CTImageStorage


class ValidationError(ValueError):
    """A CT/SEG pair cannot be interpreted safely within this tutorial's scope."""


def require(condition: bool, message: str) -> None:
    if not condition:
        raise ValidationError(message)


def numeric_vector(value, length: int, name: str) -> np.ndarray:
    result = np.asarray(value, dtype=np.float64)
    require(result.shape == (length,) and np.isfinite(result).all(), f"Invalid {name}")
    return result

### Load the CT volume with highdicom

In [ ]:
def load_ct_volume(datasets, expected_count: int):
    """Check identity/completeness and delegate volume geometry and HU to highdicom."""
    require(len(datasets) == expected_count, "CT instance count differs from the catalog")
    require(len(datasets) >= 2, "A volume needs at least two CT slices")
    first = datasets[0]
    uids = [str(ds.SOPInstanceUID) for ds in datasets]
    require(len(set(uids)) == len(uids), "Duplicate CT SOP Instance UID")
    for ds in datasets:
        require(str(ds.SOPClassUID) == str(CTImageStorage), "Only legacy single-frame CT is supported")
        require(int(ds.get("NumberOfFrames", 1)) == 1, "Enhanced/multiframe CT is unsupported")
        require(ds.Modality == "CT", "Non-CT instance in the CT series")
        for key in ("PatientID", "StudyInstanceUID", "SeriesInstanceUID", "FrameOfReferenceUID"):
            require(str(ds.get(key, "")) == str(first.get(key, "")) != "", f"Inconsistent CT {key}")
        require("ModalityLUTSequence" not in ds, "This example supports linear CT rescaling only")
        # DICOM PS3.3 C.8.2.1: original non-localizer, non-multi-energy CT is HU,
        # and RescaleType is optional in that case. Do not infer units for derived CT.
        image_type = list(ds.get("ImageType", []))
        implicit_hu = (len(image_type) >= 3 and image_type[0] == "ORIGINAL"
                       and image_type[2] != "LOCALIZER"
                       and str(ds.get("MultienergyCTAcquisition", "NO")) == "NO")
        units = str(ds.get("RescaleType", "HU" if implicit_hu else "")).upper()
        require(units == "HU", "CT rescale units cannot be verified as HU")
        require("RescaleSlope" in ds and "RescaleIntercept" in ds, "CT rescale parameters are missing")
        slope, intercept = float(ds.RescaleSlope), float(ds.RescaleIntercept)
        require(math.isfinite(slope) and math.isfinite(intercept) and slope != 0, "Invalid CT rescale parameters")
    try:
        volume = hd.get_volume_from_series(
            datasets, dtype=np.float32, apply_real_world_transform=False,
            apply_modality_transform=True, apply_voi_transform=False,
            apply_presentation_lut=False, atol=0.01,
        )
    except (ValueError, RuntimeError) as error:
        raise ValidationError(f"highdicom CT volume validation failed: {error}") from error
    require(np.isfinite(volume.array).all(), "Non-finite CT intensities after rescaling")
    # Use the library's affine to record provenance in the exported slice order.
    positions = np.asarray([ds.ImagePositionPatient for ds in datasets], dtype=np.float64)
    slice_indices = volume.map_reference_to_indices(positions)[:, 0]
    ordered = [datasets[int(i)] for i in np.argsort(slice_indices, kind="stable")]
    return volume, ordered


def select_catalog_pairs(index_table, seg_table, geometry_table, collection, n_patients, max_dicom_mb):
    """Select referenced regular CT/SEG pairs before downloading any DICOM."""
    require(isinstance(n_patients, int) and n_patients >= 2, "At least two patient pairs are required")
    require(math.isfinite(max_dicom_mb) and max_dicom_mb > 0, "Invalid DICOM size cap")
    cohort = index_table.loc[index_table.collection_id.eq(collection)].copy()
    seg_rows = cohort.loc[cohort.Modality.eq("SEG") & cohort.instanceCount.eq(1)].merge(
        seg_table[["SeriesInstanceUID", "segmented_SeriesInstanceUID", "SegmentationType"]],
        on="SeriesInstanceUID", validate="one_to_one")
    seg_rows = seg_rows.loc[seg_rows.SegmentationType.eq("BINARY")]
    eligible_geometry = geometry_table.loc[
        geometry_table.regularly_spaced_3d_volume.eq(True).fillna(False),
        ["SeriesInstanceUID", "regularly_spaced_3d_volume"],
    ]
    ct_rows = cohort.loc[cohort.Modality.eq("CT")].merge(
        eligible_geometry, on="SeriesInstanceUID", validate="one_to_one")
    pairs = seg_rows.merge(ct_rows, left_on="segmented_SeriesInstanceUID", right_on="SeriesInstanceUID",
                           suffixes=("_seg", "_ct"), validate="many_to_one")
    pairs = pairs.loc[pairs.PatientID_seg.eq(pairs.PatientID_ct)
                      & pairs.StudyInstanceUID_seg.eq(pairs.StudyInstanceUID_ct)
                      & pairs.license_short_name_seg.eq("CC BY 3.0")
                      & pairs.license_short_name_ct.eq("CC BY 3.0")].copy()
    pairs["estimated_mb"] = pairs.series_size_MB_seg + pairs.series_size_MB_ct
    pairs = pairs.sort_values(["estimated_mb", "PatientID_ct", "SeriesInstanceUID_seg"], kind="stable")
    pairs = pairs.drop_duplicates("PatientID_ct").head(n_patients)
    require(len(pairs) == n_patients, "Not enough eligible patient pairs with regular CT geometry")
    require(np.isfinite(pairs.estimated_mb).all() and (pairs.estimated_mb > 0).all(), "Invalid size estimates")
    require(float(pairs.estimated_mb.sum()) <= max_dicom_mb, "Cohort exceeds MAX_DICOM_MB; reduce N_PATIENTS")
    return pairs

### Check every SEG frame against its referenced CT

In [ ]:
def functional_group(frame, shared, sequence_name):
    sequence = frame.get(sequence_name, shared.get(sequence_name, None))
    require(sequence is not None and len(sequence) == 1, f"Missing/ambiguous SEG {sequence_name}")
    return sequence[0]


def validate_seg_geometry(seg, ct_datasets, allow_omitted_empty_frames: bool = False):
    """Verify source references and every SEG frame on the CT grid before decoding."""
    require(seg.Modality == "SEG" and str(seg.SegmentationType) == "BINARY", "Only BINARY DICOM SEG is supported")
    ct = ct_datasets[0]
    for key in ("PatientID", "StudyInstanceUID", "FrameOfReferenceUID"):
        require(str(seg.get(key, "")) == str(ct.get(key, "")) != "", f"CT/SEG {key} mismatch")
    require((seg.Rows, seg.Columns) == (ct.Rows, ct.Columns), "CT/SEG dimensions differ")
    series_refs = seg.get("ReferencedSeriesSequence", [])
    require(len(series_refs) == 1, "SEG must reference exactly one source series")
    require(str(series_refs[0].SeriesInstanceUID) == str(ct.SeriesInstanceUID), "SEG references a different CT series")
    referenced_instances = series_refs[0].get("ReferencedInstanceSequence", [])
    referenced_uids = [str(item.ReferencedSOPInstanceUID) for item in referenced_instances]
    require(len(referenced_uids) == len(set(referenced_uids)), "Duplicate SEG source reference")
    ct_by_uid = {str(ds.SOPInstanceUID): ds for ds in ct_datasets}
    require(set(referenced_uids) == set(ct_by_uid), "SEG source references and complete CT series do not match")
    require(all(str(item.ReferencedSOPClassUID) == str(CTImageStorage) for item in referenced_instances), "SEG references unsupported source classes")
    shared_sequence = seg.get("SharedFunctionalGroupsSequence", [])
    require(len(shared_sequence) == 1, "SEG needs one Shared Functional Groups item")
    shared = shared_sequence[0]
    frames = seg.get("PerFrameFunctionalGroupsSequence", [])
    require(len(frames) == int(seg.NumberOfFrames), "SEG functional-group frame count mismatch")
    segment_numbers = [int(item.SegmentNumber) for item in seg.SegmentSequence]
    require(segment_numbers and len(set(segment_numbers)) == len(segment_numbers), "Invalid SEG segment numbering")
    seen = set()
    preserved_flags = []
    for frame in frames:
        derivations = frame.get("DerivationImageSequence", shared.get("DerivationImageSequence", []))
        sources = [source for derivation in derivations for source in derivation.get("SourceImageSequence", [])]
        require(len(sources) == 1, "Each SEG frame must reference one single-frame CT instance")
        source = sources[0]
        uid = str(source.ReferencedSOPInstanceUID)
        require(uid in ct_by_uid, "SEG frame references a CT instance that is missing")
        require(str(source.ReferencedSOPClassUID) == str(CTImageStorage), "SEG frame source is not single-frame CT")
        require("ReferencedFrameNumber" not in source, "Multiframe source references are unsupported")
        preserved = str(source.get("SpatialLocationsPreserved", ""))
        require(preserved in ("", "YES"), "SEG explicitly does not preserve spatial locations")
        preserved_flags.append(preserved)
        source_ct = ct_by_uid[uid]
        orientation = functional_group(frame, shared, "PlaneOrientationSequence").ImageOrientationPatient
        position = functional_group(frame, shared, "PlanePositionSequence").ImagePositionPatient
        pixel_spacing = functional_group(frame, shared, "PixelMeasuresSequence").PixelSpacing
        require(np.allclose(numeric_vector(orientation, 6, "SEG orientation"), source_ct.ImageOrientationPatient, rtol=0, atol=1e-4), "SEG orientation differs from its source CT")
        require(np.allclose(numeric_vector(position, 3, "SEG position"), source_ct.ImagePositionPatient, rtol=0, atol=0.01), "SEG position differs from its source CT")
        require(np.allclose(numeric_vector(pixel_spacing, 2, "SEG pixel spacing"), source_ct.PixelSpacing, rtol=0, atol=1e-4), "SEG pixel spacing differs from its source CT")
        number = int(functional_group(frame, shared, "SegmentIdentificationSequence").ReferencedSegmentNumber)
        require(number in segment_numbers, "SEG frame names an undefined segment")
        require((uid, number) not in seen, "Duplicate SEG frame for one source instance and segment")
        seen.add((uid, number))
    missing = {(uid, number) for uid in ct_by_uid for number in segment_numbers} - seen
    require(not missing or allow_omitted_empty_frames, "SEG omits source/segment frames; explicit empty-frame policy is required")
    return {
        "segment_numbers": segment_numbers,
        "omitted_source_segment_frames": len(missing),
        "spatial_locations_flag_absent": any(flag == "" for flag in preserved_flags),
    }

### Preserve coded segment semantics

In [ ]:
def code_metadata(code):
    return {
        "value": str(code.CodeValue),
        "scheme": str(code.CodingSchemeDesignator),
        "meaning": str(code.CodeMeaning),
    }


def segment_metadata(seg):
    result = []
    for item in seg.SegmentSequence:
        category = code_metadata(item.SegmentedPropertyCategoryCodeSequence[0])
        property_type = code_metadata(item.SegmentedPropertyTypeCodeSequence[0])
        modifiers = [code_metadata(c) for c in item.SegmentedPropertyTypeCodeSequence[0].get("SegmentedPropertyTypeModifierCodeSequence", [])]
        regions = []
        for region in item.get("AnatomicRegionSequence", []):
            regions.append({"code": code_metadata(region), "modifiers": [code_metadata(c) for c in region.get("AnatomicRegionModifierSequence", [])]})
        # Include modifiers/anatomic region in the semantic key, not just the free-text label.
        def identity(code):
            return (code["scheme"], code["value"])

        semantic_key = json.dumps({
            "category": identity(category), "type": identity(property_type),
            "type_modifiers": sorted(identity(code) for code in modifiers),
            "regions": sorted((identity(region["code"]), sorted(identity(code) for code in region["modifiers"])) for region in regions),
        }, sort_keys=True)
        result.append({
            "segment_number": int(item.SegmentNumber),
            "label": str(item.SegmentLabel),
            "description": str(item.get("SegmentDescription", "")),
            "algorithm_type": str(item.SegmentAlgorithmType),
            "algorithm_name": str(item.get("SegmentAlgorithmName", "")),
            "category": category, "property_type": property_type,
            "type_modifiers": modifiers, "anatomic_regions": regions,
            "semantic_key": semantic_key,
        })
    require(len({item["semantic_key"] for item in result}) == len(result), "Ambiguous duplicate semantic segment definitions")
    return sorted(result, key=lambda item: item["semantic_key"])

### Convert a validated pair

In [ ]:
def validate_and_convert(ct_datasets, seg, expected_count: int, allow_omitted_empty_frames: bool = False):
    """Validate one real DICOM pair; return HU image, separate masks, and provenance."""
    ct_volume, ct_datasets = load_ct_volume(ct_datasets, expected_count)
    checks = validate_seg_geometry(seg, ct_datasets, allow_omitted_empty_frames)
    segments = segment_metadata(seg)
    source_uids = [str(ds.SOPInstanceUID) for ds in ct_datasets]
    try:
        mask_volume = seg.get_volume(
            segment_numbers=[item["segment_number"] for item in segments],
            combine_segments=False, dtype=np.uint8,
            allow_missing_positions=allow_omitted_empty_frames, atol=0.01,
        ).match_geometry(ct_volume, tol=1e-4)
    except (ValueError, RuntimeError) as error:
        raise ValidationError(f"highdicom SEG grid alignment failed: {error}") from error
    masks_dhwk = mask_volume.array
    require(masks_dhwk.shape == (*ct_volume.array.shape, len(segments)), "Decoded SEG shape differs from CT")
    require(np.isin(masks_dhwk, (0, 1)).all(), "Non-binary decoded mask values")
    image_hu = np.ascontiguousarray(ct_volume.array[np.newaxis], dtype=np.float32)
    masks = np.ascontiguousarray(np.moveaxis(masks_dhwk, -1, 0))
    affine = ct_volume.affine
    checks["ct_volume_loader"] = "highdicom.get_volume_from_series"
    checks["hu_units_from_original_ct_rule"] = any("RescaleType" not in ds for ds in ct_datasets)
    checks["seg_volume_loader"] = "highdicom.seg.Segmentation.get_volume + Volume.match_geometry"
    checks["grid_match_tolerance_voxels"] = 1e-4
    first = ct_datasets[0]
    metadata = {
        "patient_id": str(first.PatientID),
        "study_uid": str(first.StudyInstanceUID),
        "ct_series_uid": str(first.SeriesInstanceUID),
        "seg_series_uid": str(seg.SeriesInstanceUID),
        "seg_sop_instance_uid": str(seg.SOPInstanceUID),
        "frame_of_reference_uid": str(first.FrameOfReferenceUID),
        "source_sop_instance_uids_in_slice_order": source_uids,
        "array_axes": ["slice", "row", "column"],
        "world_coordinate_system": "DICOM LPS, millimeters",
        "affine_dhw_to_lps_mm": affine.tolist(),
        "image_units": "HU", "segments": segments, "checks": checks,
        "image_shape": list(image_hu.shape), "mask_shape": list(masks.shape),
        "nonzero_voxels_by_channel": [int(np.count_nonzero(mask)) for mask in masks],
    }
    return image_hu, masks, metadata

### Patient grouping and source checksums

In [ ]:
def patient_disjoint_split(records, seed: int = 2026, test_fraction: float = 0.25):
    """Assign all cases for each collection/patient group to the same partition."""
    require(0 < test_fraction < 1, "test_fraction must be between 0 and 1")
    for record in records:
        require(bool(record.get("collection_id")) and bool(record.get("patient_id")), "Missing patient group identity")
    groups = sorted({(record["collection_id"], record["patient_id"]) for record in records})
    require(len(groups) >= 2, "At least two patient groups are needed")
    random.Random(seed).shuffle(groups)
    n_test = min(len(groups) - 1, max(1, math.ceil(len(groups) * test_fraction)))
    test_groups = set(groups[:n_test])
    result = [dict(record, split="test" if (record["collection_id"], record["patient_id"]) in test_groups else "train") for record in records]
    train = {(r["collection_id"], r["patient_id"]) for r in result if r["split"] == "train"}
    test = {(r["collection_id"], r["patient_id"]) for r in result if r["split"] == "test"}
    require(train and test and train.isdisjoint(test), "Patient split validation failed")
    return result


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

### 2. Load a pinned metadata catalog

The main catalog has one row per series. `seg_index` supplies the referenced
CT series, and `volume_geometry_index` supplies precomputed regular-grid checks. We compare selected cloud object identifiers with the download
client's package index, so a catalog/client mismatch fails before downloading.
The Hugging Face option downloads three Parquet metadata files and records their
SHA256 checksums; this can be larger than the small DICOM example itself.

In [ ]:
client = IDCClient()
require(client.get_idc_version() == "v25", "This example is pinned to IDC v25")
catalog_paths = {}
if CATALOG_SOURCE == "huggingface":
    cache_dir = WORK_DIR / "catalog" / HF_REVISION
    cache_dir.mkdir(parents=True, exist_ok=True)
    for name in ("idc_index", "seg_index", "volume_geometry_index"):
        path = cache_dir / f"{name}.parquet"
        if not path.exists():
            url = f"https://huggingface.co/datasets/ImagingDataCommons/idc-index-data/resolve/{HF_REVISION}/{name}.parquet"
            temporary = path.with_suffix(".partial")
            with requests.get(url, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open("wb") as stream:
                    for chunk in response.iter_content(1024 * 1024):
                        stream.write(chunk)
            temporary.replace(path)
        catalog_paths[name] = path
    index_table = pd.read_parquet(catalog_paths["idc_index"])
    seg_table = pd.read_parquet(catalog_paths["seg_index"])
    geometry_table = pd.read_parquet(catalog_paths["volume_geometry_index"])
elif CATALOG_SOURCE == "idc-index":
    client.fetch_index("seg_index")
    client.fetch_index("volume_geometry_index")
    require(getattr(client, "seg_index", None) is not None, "SEG index download failed")
    require(getattr(client, "volume_geometry_index", None) is not None, "Geometry index download failed")
    index_table, seg_table, geometry_table = client.index, client.seg_index, client.volume_geometry_index
    catalog_paths = {"idc_index": Path(client.indices_overview["index"]["file_path"]),
                     "seg_index": Path(client.indices_overview["seg_index"]["file_path"]),
                     "volume_geometry_index": Path(client.indices_overview["volume_geometry_index"]["file_path"])}
else:
    raise ValidationError("Unknown CATALOG_SOURCE")

catalog_provenance = {}
for name, path in catalog_paths.items():
    generated_version = pq.read_metadata(path).metadata[b"idc_index_data_version"].decode()
    require(generated_version.split("-")[0] == version("idc-index-data"), "Catalog release differs from the download client")
    catalog_provenance[name] = {"sha256": file_sha256(path), "generated_version": generated_version,
                                "size_bytes": path.stat().st_size}
print(f"Catalog source: {CATALOG_SOURCE}; IDC release: {client.get_idc_version()}")

### 3. Select four referenced pairs and inspect the download estimate

Restrict both modalities to this collection and the documented CC BY 3.0 license.
Require one SEG instance per series and a CT marked `regularly_spaced_3d_volume`
in the geometry index, retain one pair per patient, and show the
estimate before any DICOM download. Raise an error if the configured size cap
or patient count cannot be met. Check the [collection's usage and citation requirements](https://www.cancerimagingarchive.net/collection/c4kc-kits/).

In [ ]:
cohort = index_table.loc[index_table.collection_id.eq(COLLECTION)].copy()
pairs = select_catalog_pairs(index_table, seg_table, geometry_table, COLLECTION, N_PATIENTS, MAX_DICOM_MB)
estimated_mb = float(pairs.estimated_mb.sum())
require(estimated_mb <= MAX_DICOM_MB, "Cohort exceeds MAX_DICOM_MB; reduce N_PATIENTS")
selected_uids = pairs.SeriesInstanceUID_ct.tolist() + pairs.SeriesInstanceUID_seg.tolist()
selected_catalog = cohort.loc[cohort.SeriesInstanceUID.isin(selected_uids)]
client_rows = client.index.set_index("SeriesInstanceUID")
for row in selected_catalog.to_dict("records"):
    require(row["SeriesInstanceUID"] in client_rows.index, "Selected series missing from download client")
    download_row = client_rows.loc[row["SeriesInstanceUID"]]
    require(str(download_row.crdc_series_uuid) == str(row["crdc_series_uuid"])
            and int(download_row.instanceCount) == int(row["instanceCount"]), "Catalog/download client mismatch")
selected_catalog.to_json(OUTPUT_DIR / "cohort_catalog.json", orient="records", indent=2)
print(f"DICOM estimate: {estimated_mb:.2f} MB for {len(pairs)} patients; cap {MAX_DICOM_MB:.0f} MB")
display(pairs[["PatientID_ct", "instanceCount_ct", "estimated_mb", "license_short_name_ct"]]
        .rename(columns={"PatientID_ct": "Patient", "instanceCount_ct": "CT slices", "estimated_mb": "CT + SEG (MB)", "license_short_name_ct": "License"}))

### 4. Download, validate and export one patient at a time

Every file's SHA256 is recorded. The catalog's CT instance count must match the
downloaded series and the SEG's global source references. All source/segment
frames must be present by default; zero-filled omitted frames require an explicit
dataset assumption, controlled by `ALLOW_OMITTED_EMPTY_FRAMES`.

The preview uses a fixed HU display window and the slice with the largest total
segmented area. This changes only the display, not stored HU values. No patient
identifiers are drawn into the image. Each exported mask channel has its original
segment metadata; DICOM algorithm type is preserved without reinterpretation.

In [ ]:
records = []
canonical_channel_keys = None
for row in pairs.to_dict("records"):
    patient = row["PatientID_ct"]
    ct_uid, seg_uid = row["SeriesInstanceUID_ct"], row["SeriesInstanceUID_seg"]
    client.download_from_selection(downloadDir=str(DICOM_DIR), seriesInstanceUID=[ct_uid, seg_uid],
                                   show_progress_bar=False, use_s5cmd_sync=True)
    study_dir = DICOM_DIR / COLLECTION / patient / row["StudyInstanceUID_ct"]
    ct_paths = sorted((study_dir / f"CT_{ct_uid}").glob("*.dcm"))
    seg_paths = sorted((study_dir / f"SEG_{seg_uid}").glob("*.dcm"))
    require(len(seg_paths) == 1, "Expected exactly one downloaded SEG instance")
    ct_datasets = [pydicom.dcmread(path) for path in ct_paths]
    seg = hd.seg.segread(seg_paths[0])
    require(all(str(ds.SeriesInstanceUID) == ct_uid and str(ds.PatientID) == patient for ds in ct_datasets), "Downloaded CT differs from selection")
    require(str(seg.SeriesInstanceUID) == seg_uid, "Downloaded SEG differs from selection")
    image_hu, masks, metadata = validate_and_convert(ct_datasets, seg, int(row["instanceCount_ct"]),
                                                    ALLOW_OMITTED_EMPTY_FRAMES)
    channel_keys = [item["semantic_key"] for item in metadata["segments"]]
    if canonical_channel_keys is None:
        canonical_channel_keys = channel_keys
    require(channel_keys == canonical_channel_keys, "Cases have different semantic channel definitions")
    sample_id = hashlib.sha256(ct_uid.encode()).hexdigest()[:16]
    tensor_file = f"{sample_id}.pt"
    torch.save({"image_hu": torch.from_numpy(image_hu), "masks": torch.from_numpy(masks)}, OUTPUT_DIR / tensor_file)
    metadata["collection_id"] = COLLECTION
    metadata["catalog_regularly_spaced_3d_volume"] = bool(row["regularly_spaced_3d_volume"])
    metadata["sources"] = [{"path": str(path.relative_to(DICOM_DIR)), "sha256": file_sha256(path),
                            "size_bytes": path.stat().st_size} for path in ct_paths + seg_paths]
    metadata["license_ct"] = row["license_short_name_ct"]
    metadata["license_seg"] = row["license_short_name_seg"]
    metadata["source_dois"] = sorted({row["source_DOI_ct"], row["source_DOI_seg"]})
    metadata_file = f"{sample_id}.json"
    (OUTPUT_DIR / metadata_file).write_text(json.dumps(metadata, indent=2))
    records.append({"collection_id": COLLECTION, "patient_id": patient,
                    "ct_series_uid": ct_uid, "seg_series_uid": seg_uid,
                    "tensor_file": tensor_file, "metadata_file": metadata_file,
                    "ct_slices": image_hu.shape[1], "image_shape": list(image_hu.shape),
                    "mask_shape": list(masks.shape), "nonzero_voxels": metadata["nonzero_voxels_by_channel"],
                    "dicom_bytes": sum(item["size_bytes"] for item in metadata["sources"])})
    slice_index = int(np.argmax(masks.sum(axis=(0, 2, 3))))
    fig, axes = plt.subplots(1, len(masks) + 1, figsize=(5 * (len(masks) + 1), 4))
    axes[0].imshow(image_hu[0, slice_index], cmap="gray", vmin=-200, vmax=300)
    axes[0].set_title(f"CT (HU); slice {slice_index}")
    for axis, mask, segment in zip(axes[1:], masks, metadata["segments"]):
        axis.imshow(image_hu[0, slice_index], cmap="gray", vmin=-200, vmax=300)
        axis.imshow(np.ma.masked_equal(mask[slice_index], 0), cmap="autumn", alpha=0.45, vmin=0, vmax=1)
        axis.set_title(f"{segment['label']} — {segment['algorithm_type']}")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(f"Validated CT/SEG pair · {sample_id}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{sample_id}_overlay.png", dpi=120)
    plt.show()
    plt.close(fig)
    print(f"Validated {patient}: {image_hu.shape} image; {masks.shape} masks; omitted frames {metadata['checks']['omitted_source_segment_frames']}")
    del ct_datasets, seg, image_hu, masks

### 5. Write and verify the patient-disjoint split

Grouping uses `(collection_id, PatientID)`, since PatientID is scoped to a
collection. The fixed seed is applied to a sorted patient list, so input row
order does not affect assignments. All cases for a patient share one partition.
This grouping cannot identify aliases or duplicate people across collections.

In [ ]:
split_records = patient_disjoint_split(records, SPLIT_SEED, TEST_FRACTION)
manifest = {
    "idc_release": client.get_idc_version(), "catalog_source": CATALOG_SOURCE,
    "hf_revision": HF_REVISION if CATALOG_SOURCE == "huggingface" else None,
    "catalog_provenance": catalog_provenance,
    "packages": {name: version(name) for name in ("idc-index", "idc-index-data", "highdicom", "pydicom", "numpy", "pandas", "torch", "matplotlib")},
    "split_seed": SPLIT_SEED, "test_fraction": TEST_FRACTION,
    "sampling": "smallest referenced regular-grid CT/SEG pair per patient; tutorial convenience sample",
    "geometry_preflight": "volume_geometry_index.regularly_spaced_3d_volume",
    "allow_omitted_empty_frames": ALLOW_OMITTED_EMPTY_FRAMES,
    "channel_semantic_keys": canonical_channel_keys,
    "estimated_dicom_mb": estimated_mb,
    "actual_dicom_bytes": sum(record["dicom_bytes"] for record in records),
    "runtime_seconds": round(perf_counter() - run_started, 2),
    "records": split_records,
}
(OUTPUT_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2))
display(pd.DataFrame(split_records)[["patient_id", "split", "ct_slices", "image_shape", "mask_shape", "nonzero_voxels"]])
print(f"Actual selected DICOM: {manifest['actual_dicom_bytes'] / 1e6:.2f} MB; runtime {manifest['runtime_seconds']:.2f} seconds")

### 6. Consume saved tensors with PyTorch

This Dataset opens one case at a time. Batch size 1 accommodates different
volume depths. The optional fixed window `[-200, 300] HU -> [0, 1]` is an
illustrative input transform, not a recommended training configuration or a
transformation fitted on the test set. Stored tensors always retain HU.
Only load `.pt` files generated by this notebook, using `weights_only=True`.

In [ ]:
class CTSegDataset(torch.utils.data.Dataset):
    def __init__(self, output_dir, partition, normalize=False):
        require(partition in ("train", "test"), "Unknown dataset partition")
        self.output_dir = Path(output_dir)
        saved_manifest = json.loads((self.output_dir / "split_manifest.json").read_text())
        self.records = [record for record in saved_manifest["records"] if record["split"] == partition]
        self.normalize = normalize

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        tensors = torch.load(self.output_dir / record["tensor_file"], map_location="cpu", weights_only=True)
        image = tensors["image_hu"]
        if self.normalize:
            image = (image.clamp(-200, 300) + 200) / 500
        return {"image": image, "masks": tensors["masks"], "patient_id": record["patient_id"]}

train_dataset = CTSegDataset(OUTPUT_DIR, "train", normalize=True)
test_dataset = CTSegDataset(OUTPUT_DIR, "test", normalize=True)
loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
print("Train/test patients:", len(train_dataset), len(test_dataset))
print("Batch shapes:", tuple(batch["image"].shape), tuple(batch["masks"].shape))

## Checks

Reload saved files and verify shapes, binary values, coded channel consistency,
source count, and patient separation. These checks demonstrate data preparation;
they do not establish annotation quality or predict segmentation accuracy.

In [ ]:
saved = json.loads((OUTPUT_DIR / "split_manifest.json").read_text())
train_groups = {(r["collection_id"], r["patient_id"]) for r in saved["records"] if r["split"] == "train"}
test_groups = {(r["collection_id"], r["patient_id"]) for r in saved["records"] if r["split"] == "test"}
require(train_groups and test_groups and train_groups.isdisjoint(test_groups), "Saved split leaks patients")
for record in saved["records"]:
    tensors = torch.load(OUTPUT_DIR / record["tensor_file"], map_location="cpu", weights_only=True)
    metadata = json.loads((OUTPUT_DIR / record["metadata_file"]).read_text())
    require(list(tensors["image_hu"].shape) == record["image_shape"], "Saved image shape mismatch")
    require(list(tensors["masks"].shape) == record["mask_shape"], "Saved mask shape mismatch")
    require(tensors["image_hu"].dtype == torch.float32 and torch.isfinite(tensors["image_hu"]).all().item(), "Invalid HU tensor")
    require(tensors["masks"].dtype == torch.uint8 and torch.all((tensors["masks"] == 0) | (tensors["masks"] == 1)).item(), "Invalid binary masks")
    require([segment["semantic_key"] for segment in metadata["segments"]] == saved["channel_semantic_keys"], "Saved channels differ between cases")
    require(len(metadata["source_sop_instance_uids_in_slice_order"]) == record["ct_slices"], "Saved source count mismatch")
    require(metadata["catalog_regularly_spaced_3d_volume"] is True, "Missing catalog geometry preflight")
    del tensors
print("PASS: validated real CT/SEG pairs, consistent tensor channels, and disjoint patient groups")

## Next Steps

Use a larger, representative cohort before training or assessing a model.
Keep patient grouping when adding studies, slices or patches. Fit any learned
preprocessing on training patients only. Validate resampling and class semantics
explicitly before extending this example to other collections or modalities.

Outputs are under `idc_ct_seg_data/outputs`: `.pt` tensors, per-case JSON geometry
and source checksums, overlay PNGs, the selected catalog rows, and the split
manifest. The four-patient split is an educational artifact, not a benchmark.

## Support

For questions, use the [IDC user forum](https://discourse.canceridc.dev) or open
an issue in [IDC-Tutorials](https://github.com/ImagingDataCommons/IDC-Tutorials/issues).
More examples are available in the [IDC-Tutorials repository](https://github.com/ImagingDataCommons/IDC-Tutorials).

## Acknowledgments

Imaging Data Commons has been funded in whole or in part with Federal funds from
the National Cancer Institute, National Institutes of Health, under Task Order
No. HHSN26110071 under Contract No. HHSN261201500003I.

If you use IDC in research, cite Fedorov et al., *National Cancer Institute Imaging
Data Commons: Toward Transparency, Reproducibility, and Scalability in Imaging
Artificial Intelligence*, RadioGraphics (2023), [doi:10.1148/rg.230180](https://doi.org/10.1148/rg.230180).

The C4KC-KiTS data are provided under CC BY 3.0. Cite Heller et al., *Data from
C4KC-KiTS*, The Cancer Imaging Archive (2019), [doi:10.7937/TCIA.2019.IX49E8NX](https://doi.org/10.7937/TCIA.2019.IX49E8NX),
and follow the [collection's full citation instructions](https://www.cancerimagingarchive.net/collection/c4kc-kits/),
including Heller et al., *The KiTS19 Challenge Data: 300 Kidney Tumor Cases with
Clinical Context, CT Semantic Segmentations, and Surgical Outcomes*,
[arXiv:1904.00445](https://arxiv.org/abs/1904.00445).

Libraries used: [idc-index](https://github.com/ImagingDataCommons/idc-index),
[highdicom](https://highdicom.readthedocs.io/en/latest/seg.html),
[pydicom](https://pydicom.github.io/), and [PyTorch](https://pytorch.org/).